 该项目主要是利用大模型微调完成中医药命名实体识别（NER）的任务


In [26]:
import json

def process_data(input_file_path, output_file_path):
    with open(input_file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    dataset = []
    current_input = []
    current_output = []

    for line in lines:
        line = line.strip()
        if line:
            parts = line.split(' ')
            if len(parts) == 2:
                char, label = parts
                current_input.append(char)
                current_output.append(label)
            else:
                print(f"错误：数据格式不正确，跳过行：'{line}'")
        else:
            if current_input and current_output:
                dataset.append({
                    "instruction": "请对以下文本进行命名实体识别，输出每个字符的BIO标注。B表示实体的开始，I表示实体的内部，O表示非实体部分。最后以列表的格式输出结果：\n",
                    "input": str(current_input),
                    "output": str(current_output)
                })
                # 重置当前组数据
                current_input = []
                current_output = []

    # 处理最后一组数据（如果没有空行结尾）
    if current_input and current_output:
        dataset.append({
            "instruction": "请对以下文本进行命名实体识别，输出每个字符的BIO标注。B表示实体的开始，I表示实体的内部，O表示非实体部分。最后以列表的格式输出结果：\n",
            "input": str(current_input),
            "output": str(current_output)
        })

    # 保存数据
    with open(output_file_path, 'w', encoding='utf-8') as file:
        json.dump(dataset, file, ensure_ascii=False, indent=2)

    print(f"处理完成，新的数据集已保存到 {output_file_path}")


In [ ]:
# 生成处理后的数据集
input_file_path_train = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.train"
output_file_path_train = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical.train"
input_file_path_test = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.test"
output_file_path_test = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical.test"
input_file_path_dev = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset_raw/medical.dev"
output_file_path_dev = "/root/LLM-bootcamp/hw3/name_identify_sft/dataset/medical.dev"

process_data(input_file_path_train, output_file_path_train)
process_data(input_file_path_test, output_file_path_test)
process_data(input_file_path_dev, output_file_path_dev)

In [ ]:
# 选用基座大模型Qwen2.5-7B model
# 封装函数 
def load_model():
    